# 🧠 Build a State-of-the-Art Generative AI (LLM) From Scratch
### Architected for Google Colab & Kaggle with GPU Acceleration
This notebook implements an enterprise-grade **Generative Decoder-Only Transformer (GPT/LLaMA-style)** strictly from scratch using PyTorch.

#### 🚀 Cutting-Edge Architectural Components Built Inside:
1. **Rotary Positional Embeddings (RoPE)** - Exact relative position calculation without fixed learned offsets.
2. **RMSNorm (Root Mean Square Layer Normalization)** - High performance and training stability.
3. **SwiGLU Activations** - Modern gating mechanism outperforming standard GELU/ReLU.
4. **Multi-Head Causal Self-Attention** with real-time attention weight extraction & heatmaps.
5. **Weight Tying** between embedding table and language model classification head.
6. **Cosine Annealing Learning Rate Scheduler with Linear Warmup**.
7. **Full Visual Suite**: Dynamic loss convergence graphs and self-attention heatmaps.


In [1]:
!pip install --quiet matplotlib seaborn tqdm 
import os ,math ,time 
import torch 
import torch .nn as nn 
from torch .nn import functional as F 
import matplotlib .pyplot as plt 
import seaborn as sns 
from tqdm import tqdm 
device ='cuda'if torch .cuda .is_available ()else 'cpu'
print (f'[✓] Active Compute Device: {device }')
if device =='cuda':
    print (f'[✓] GPU Model: {torch .cuda .get_device_name (0 )}')
    print (f'[✓] Available VRAM: {torch .cuda .get_device_properties (0 ).total_memory /1e9 :.2f} GB')
else :
    print ('[!] GPU not detected. Running on CPU. (In Colab: Runtime -> Change runtime type -> T4 GPU)')



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'torch'

In [ ]:
class Config :
    vocab_size :int =50257 
    n_embd :int =384 
    n_head :int =6 
    n_layer :int =6 
    block_size :int =128 
    dropout :float =0.1 
    bias :bool =False 
    batch_size :int =32 
    learning_rate :float =6e-4 
    min_lr :float =6e-5 
    warmup_iters :int =150 
    max_iters :int =1200 
    eval_interval :int =150 
    eval_iters :int =30 
    weight_decay :float =0.1 
    grad_clip :float =1.0 
    seed :int =42 
cfg =Config ()
torch .manual_seed (cfg .seed )
if torch .cuda .is_available ():
    torch .cuda .manual_seed_all (cfg .seed )
print ('[✓] Configuration initialized successfully.')


In [ ]:
class RotaryEmbedding (nn .Module ):
    def __init__ (self ,dim :int ,max_seq_len :int =2048 ):
        super ().__init__ ()
        self .dim =dim 
        inv_freq =1.0 /(10000 **(torch .arange (0 ,dim ,2 ).float ()/dim ))
        self .register_buffer ('inv_freq',inv_freq )
        t =torch .arange (max_seq_len ,dtype =torch .float32 )
        freqs =torch .outer (t ,self .inv_freq )
        emb =torch .cat ((freqs ,freqs ),dim =-1 )
        self .register_buffer ('cos_cached',emb .cos (),persistent =False )
        self .register_buffer ('sin_cached',emb .sin (),persistent =False )
    def rotate_half (self ,x ):
        x1 ,x2 =x [...,:self .dim //2 ],x [...,self .dim //2 :]
        return torch .cat ((-x2 ,x1 ),dim =-1 )
    def forward (self ,x ,seq_len ):
        cos =self .cos_cached [:seq_len ,:].unsqueeze (0 ).unsqueeze (0 )
        sin =self .sin_cached [:seq_len ,:].unsqueeze (0 ).unsqueeze (0 )
        return (x *cos )+(self .rotate_half (x )*sin )
class RMSNorm (nn .Module ):
    def __init__ (self ,dim :int ,eps :float =1e-6 ):
        super ().__init__ ()
        self .eps =eps 
        self .weight =nn .Parameter (torch .ones (dim ))
    def forward (self ,x ):
        variance =x .pow (2 ).mean (-1 ,keepdim =True )
        return x *torch .rsqrt (variance +self .eps )*self .weight 
class CausalSelfAttention (nn .Module ):
    def __init__ (self ,config :Config ):
        super ().__init__ ()
        self .n_head =config .n_head 
        self .head_dim =config .n_embd //config .n_head 
        self .qkv_proj =nn .Linear (config .n_embd ,3 *config .n_embd ,bias =config .bias )
        self .out_proj =nn .Linear (config .n_embd ,config .n_embd ,bias =config .bias )
        self .attn_dropout =nn .Dropout (config .dropout )
        self .resid_dropout =nn .Dropout (config .dropout )
        self .rope =RotaryEmbedding (self .head_dim ,max_seq_len =config .block_size )
        self .register_buffer (
        'causal_mask',
        torch .tril (torch .ones (config .block_size ,config .block_size )).view (
        1 ,1 ,config .block_size ,config .block_size 
        ),
        )
        self .last_attn_weights =None 
    def forward (self ,x ,return_attn :bool =False ):
        B ,T ,C =x .size ()
        qkv =self .qkv_proj (x )
        q ,k ,v =qkv .chunk (3 ,dim =-1 )
        q =q .view (B ,T ,self .n_head ,self .head_dim ).transpose (1 ,2 )
        k =k .view (B ,T ,self .n_head ,self .head_dim ).transpose (1 ,2 )
        v =v .view (B ,T ,self .n_head ,self .head_dim ).transpose (1 ,2 )
        q =self .rope (q ,T )
        k =self .rope (k ,T )
        scores =torch .matmul (q ,k .transpose (-2 ,-1 ))/math .sqrt (self .head_dim )
        scores =scores .masked_fill (self .causal_mask [:,:,:T ,:T ]==0 ,float ('-inf'))
        attn_weights =F .softmax (scores ,dim =-1 )
        if return_attn or not self .training :
            self .last_attn_weights =attn_weights .detach ().cpu ()
        y =torch .matmul (self .attn_dropout (attn_weights ),v )
        y =y .transpose (1 ,2 ).contiguous ().view (B ,T ,C )
        return self .resid_dropout (self .out_proj (y ))
class FeedForward (nn .Module ):
    def __init__ (self ,config :Config ):
        super ().__init__ ()
        hidden_dim =int (4 *config .n_embd *(2 /3 ))
        self .w1 =nn .Linear (config .n_embd ,hidden_dim ,bias =config .bias )
        self .w2 =nn .Linear (hidden_dim ,config .n_embd ,bias =config .bias )
        self .w3 =nn .Linear (config .n_embd ,hidden_dim ,bias =config .bias )
        self .dropout =nn .Dropout (config .dropout )
    def forward (self ,x ):
        return self .dropout (self .w2 (F .silu (self .w1 (x ))*self .w3 (x )))
class TransformerBlock (nn .Module ):
    def __init__ (self ,config :Config ):
        super ().__init__ ()
        self .norm1 =RMSNorm (config .n_embd )
        self .attn =CausalSelfAttention (config )
        self .norm2 =RMSNorm (config .n_embd )
        self .ffn =FeedForward (config )
    def forward (self ,x ,return_attn :bool =False ):
        x =x +self .attn (self .norm1 (x ),return_attn =return_attn )
        x =x +self .ffn (self .norm2 (x ))
        return x 
class MiniLLM (nn .Module ):
    def __init__ (self ,config :Config ):
        super ().__init__ ()
        self .config =config 
        self .tok_embeddings =nn .Embedding (config .vocab_size ,config .n_embd )
        self .drop =nn .Dropout (config .dropout )
        self .blocks =nn .ModuleList ([TransformerBlock (config )for _ in range (config .n_layer )])
        self .norm_f =RMSNorm (config .n_embd )
        self .lm_head =nn .Linear (config .n_embd ,config .vocab_size ,bias =False )
        self .tok_embeddings .weight =self .lm_head .weight 
        self .apply (self ._init_weights )
    def _init_weights (self ,module ):
        if isinstance (module ,nn .Linear ):
            torch .nn .init .normal_ (module .weight ,mean =0.0 ,std =0.02 )
            if module .bias is not None :
                torch .nn .init .zeros_ (module .bias )
        elif isinstance (module ,nn .Embedding ):
            torch .nn .init .normal_ (module .weight ,mean =0.0 ,std =0.02 )
    def forward (self ,idx ,targets =None ,return_attn =False ):
        B ,T =idx .size ()
        x =self .drop (self .tok_embeddings (idx ))
        for block in self .blocks :
            x =block (x ,return_attn =return_attn )
        x =self .norm_f (x )
        if targets is not None :
            logits =self .lm_head (x )
            loss =F .cross_entropy (logits .view (-1 ,logits .size (-1 )),targets .view (-1 ),ignore_index =-1 )
        else :
            logits =self .lm_head (x [:,[-1 ],:])
            loss =None 
        return logits ,loss 
    @torch .no_grad ()
    def generate (self ,idx ,max_new_tokens :int =100 ,temperature :float =0.8 ,top_k :int =40 ):
        self .eval ()
        for _ in range (max_new_tokens ):
            idx_cond =idx if idx .size (1 )<=self .config .block_size else idx [:,-self .config .block_size :]
            logits ,_ =self (idx_cond )
            logits =logits [:,-1 ,:]/max (temperature ,1e-5 )
            if top_k is not None :
                v ,_ =torch .topk (logits ,min (top_k ,logits .size (-1 )))
                logits [logits <v [:,[-1 ]]]=-float ('Inf')
            probs =F .softmax (logits ,dim =-1 )
            idx_next =torch .multinomial (probs ,num_samples =1 )
            idx =torch .cat ((idx ,idx_next ),dim =1 )
        return idx 
print ('[✓] Architecture classes compiled successfully.')


In [ ]:
import urllib .request 
data_url ='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
data_file ='dataset.txt'
if not os .path .exists (data_file ):
    print ('[*] Downloading training corpus...')
    urllib .request .urlretrieve (data_url ,data_file )
with open (data_file ,'r',encoding ='utf-8')as f :
    raw_text =f .read ()
print (f'[✓] Corpus loaded: {len (raw_text ):,} characters')
class TextDataset :
    def __init__ (self ,text :str ,block_size :int ):
        self .block_size =block_size 
        self .chars =sorted (list (set (text )))
        self .vocab_size =len (self .chars )
        self .char_to_ix ={ch :i for i ,ch in enumerate (self .chars )}
        self .ix_to_char ={i :ch for i ,ch in enumerate (self .chars )}
        data =torch .tensor ([self .char_to_ix [ch ]for ch in text ],dtype =torch .long )
        n =int (0.9 *len (data ))
        self .train_data ,self .val_data =data [:n ],data [n :]
    def encode (self ,s :str ):
        return [self .char_to_ix [c ]for c in s if c in self .char_to_ix ]
    def decode (self ,l ):
        return ''.join ([self .ix_to_char [i ]for i in l ])
    def get_batch (self ,split :str ,batch_size :int ,dev :str ):
        data =self .train_data if split =='train'else self .val_data 
        ix =torch .randint (len (data )-self .block_size ,(batch_size ,))
        x =torch .stack ([data [i :i +self .block_size ]for i in ix ])
        y =torch .stack ([data [i +1 :i +1 +self .block_size ]for i in ix ])
        return x .to (dev ),y .to (dev )
dataset =TextDataset (raw_text ,cfg .block_size )
cfg .vocab_size =dataset .vocab_size 
print (f'[✓] Distinct Vocabulary Size: {dataset .vocab_size }')


In [ ]:
model =MiniLLM (cfg ).to (device )
total_params =sum (p .numel ()for p in model .parameters ())
print (f'[✓] Model Instantiated on {device .upper ()}')
print (f'[✓] Total Parameters: {total_params :,} (~{total_params /1e6 :.2f}M)')
optimizer =torch .optim .AdamW (model .parameters (),lr =cfg .learning_rate ,weight_decay =cfg .weight_decay )
def get_lr (it :int ,cfg :Config ):
    if it <cfg .warmup_iters :
        return cfg .learning_rate *(it +1 )/(cfg .warmup_iters +1 )
    if it >cfg .max_iters :
        return cfg .min_lr 
    decay_ratio =(it -cfg .warmup_iters )/(cfg .max_iters -cfg .warmup_iters )
    coeff =0.5 *(1.0 +math .cos (math .pi *decay_ratio ))
    return cfg .min_lr +coeff *(cfg .learning_rate -cfg .min_lr )


In [ ]:
train_history ,val_history ,steps =[],[],[]
start_time =time .time ()
print ('[*] Training started. Monitor loss decrease below:')
model .train ()
for it in range (cfg .max_iters ):
    lr =get_lr (it ,cfg )
    for param_group in optimizer .param_groups :
        param_group ['lr']=lr 
    xb ,yb =dataset .get_batch ('train',cfg .batch_size ,device )
    logits ,loss =model (xb ,yb )
    optimizer .zero_grad (set_to_none =True )
    loss .backward ()
    nn .utils .clip_grad_norm_ (model .parameters (),cfg .grad_clip )
    optimizer .step ()
    if (it +1 )%cfg .eval_interval ==0 or it ==cfg .max_iters -1 :
        model .eval ()
        with torch .no_grad ():
            val_loss_acc =0.0 
            for _ in range (cfg .eval_iters ):
                xv ,yv =dataset .get_batch ('val',cfg .batch_size ,device )
                _ ,v_loss =model (xv ,yv )
                val_loss_acc +=v_loss .item ()
            val_loss_acc /=cfg .eval_iters 
        train_history .append (loss .item ())
        val_history .append (val_loss_acc )
        steps .append (it +1 )
        elapsed =time .time ()-start_time 
        print (f'Iter {it +1 :4d}/{cfg .max_iters } | LR: {lr :.5f} | Train Loss: {loss .item ():.4f} | Val Loss: {val_loss_acc :.4f} | Time: {elapsed :.1f}s')
        model .train ()
print (f'\n[✓] Training complete in {time .time ()-start_time :.1f} seconds!')


In [ ]:
plt .figure (figsize =(10 ,5 ),dpi =150 )
plt .plot (steps ,train_history ,label ='Train Loss',color ='#2563eb',lw =2.5 ,marker ='o',markersize =4 )
plt .plot (steps ,val_history ,label ='Validation Loss',color ='#ea580c',lw =2.5 ,linestyle ='--',marker ='s',markersize =4 )
plt .title ('Deep Learning Training Dynamics: Cross-Entropy Loss over Steps',fontsize =14 ,fontweight ='bold',pad =12 )
plt .xlabel ('Iterations / Steps',fontsize =12 )
plt .ylabel ('Perplexity Loss (nats)',fontsize =12 )
plt .grid (True ,linestyle =':',alpha =0.6 )
plt .legend (fontsize =11 )
plt .tight_layout ()
plt .show ()


In [ ]:
def generate_text (prompt :str ,max_tokens :int =250 ,temperature :float =0.75 ,top_k :int =35 ):
    encoded =dataset .encode (prompt )
    if not encoded :
        print ('Prompt characters not found in vocabulary!')
        return ''
    input_ids =torch .tensor ([encoded ],dtype =torch .long ,device =device )
    generated =model .generate (input_ids ,max_new_tokens =max_tokens ,temperature =temperature ,top_k =top_k )
    result =dataset .decode (generated [0 ].tolist ())
    return result 
test_prompts =['ROMEO:','KING EDWARD:','To be, or not to be']
for prompt in test_prompts :
    print ('='*60 )
    print (f'[PROMPT]: {prompt }')
    print ('-'*60 )
    output =generate_text (prompt ,max_tokens =180 ,temperature =0.72 )
    print (output )
    print ('\n')


In [ ]:
sample_input ='ROMEO: Speak,'
tokens =list (sample_input )
input_t =torch .tensor ([dataset .encode (sample_input )],dtype =torch .long ,device =device )
model .eval ()
with torch .no_grad ():
    _ =model (input_t ,return_attn =True )
    attn =model .blocks [0 ].attn .last_attn_weights 
fig ,axes =plt .subplots (1 ,4 ,figsize =(18 ,4 ),dpi =140 )
T =len (tokens )
for head_idx in range (4 ):
    ax =axes [head_idx ]
    w =attn [0 ,head_idx ,:T ,:T ].numpy ()
    im =ax .imshow (w ,cmap ='magma',interpolation ='nearest')
    ax .set_title (f'Attention Head #{head_idx +1 }',fontsize =11 ,fontweight ='bold')
    ax .set_xticks (range (T ))
    ax .set_xticklabels (tokens ,rotation =45 ,fontsize =8 )
    ax .set_yticks (range (T ))
    ax .set_yticklabels (tokens ,fontsize =8 )
    ax .set_xlabel ('Key (Context)')
    if head_idx ==0 :
        ax .set_ylabel ('Query (Target)')
    fig .colorbar (im ,ax =ax ,fraction =0.046 ,pad =0.04 )
plt .suptitle ('Causal Self-Attention Weight Activations Across Attention Heads',fontsize =14 ,fontweight ='bold',y =1.05 )
plt .tight_layout ()
plt .show ()


In [ ]:
checkpoint_path ='scratch_llm_model.pt'
torch .save ({
'model_state_dict':model .state_dict (),
'config':cfg ,
'chars':dataset .chars ,
},checkpoint_path )
print (f'[✓] Model weights successfully exported to: {checkpoint_path }')
print (f'[✓] File Size: {os .path .getsize (checkpoint_path )/1e6 :.2f} MB')
